# 잇다 모델 A 본 학습 (Gemma 4 E4B, QLoRA)
RTX 4070 Ti 12GB, WSL(Ubuntu) 기준. 상세개발가이드 2-4, 2-6, 2-7.

**준비 (터미널에서 한 번)**
```
cd ~/projects/itda-ai && source .venv/bin/activate
uv pip install unsloth ipykernel
```
VS Code에서 이 노트북을 열고, 오른쪽 위 커널 선택에서 `.venv (Python)`을 고른 뒤 위에서부터 한 셀씩 실행.

**순서:** 모델 불러오기 → **파인튜닝 전 평가(val)** → LoRA 붙이고 학습 → **파인튜닝 후 평가(val)** → 저장 → GGUF

## 1. 경로
레포 맨 위 폴더를 자동으로 찾음. 데이터는 `ml/data/`, 결과는 `ml/train/out/`(커밋하지 않음).

In [1]:
import os, json, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "ml" / "data" / "train.jsonl").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "ml" / "data"
OUT  = ROOT / "ml" / "train" / "out_gemma_e4B"
OUT.mkdir(parents = True, exist_ok = True)
print("레포:", ROOT)
print("데이터:", sorted(p.name for p in DATA.iterdir()))   # train.jsonl, val.jsonl, synth_eval.jsonl

레포: /home/aiuser/projects/itda-ai
데이터: ['synth_eval.jsonl', 'train.jsonl', 'val.jsonl']


## 2. 모델 불러오기
4070 Ti는 bfloat16을 지원하므로 float32 전환 경고가 나오지 않아야 함.

In [2]:
from unsloth import FastModel
import torch
print("GPU:", torch.cuda.get_device_name(0), "| bf16 지원:", torch.cuda.is_bf16_supported())

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
    dtype = None,
    max_seq_length = 1024,
    load_in_4bit = True,
    full_finetuning = False,
    text_only = True,           # 추가: 이미지·오디오 부분은 불러오지 않음
    offload_embedding = True,   # 추가: 큰 임베딩(PLE)은 CPU 메모리로
)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/aiuser/projects/itda-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0929 01:12:56.967000 61622 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0929 01:12:57.004000 61622 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/home/aiuser/projects/itda-ai/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:4896: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


GPU: NVIDIA GeForce RTX 4070 Ti | bf16 지원: True


[unsloth_zoo.log|WARNING]Loading unsloth/gemma-4-E4B-it-unsloth-bnb-4bit as text-only; vision/audio towers skipped. Use FastVisionModel for multimodal inputs.


==((====))==  Unsloth 2026.9.11: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4070 Ti. Num GPUs = 1. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not planning a device map; text_only loads a decoder the repo config does not describe. Using `sequential`.


Loading weights: 100%|██████████| 719/719 [00:04<00:00, 168.80it/s]
Gemma4ForCausalLM LOAD REPORT from: unsloth/gemma-4-E4B-it-unsloth-bnb-4bit
Key                                                                          | Status     |  | 
-----------------------------------------------------------------------------+------------+--+-
model.vision_tower.encoder.layers.{0...15}.self_attn.q_norm.weight           | UNEXPECTED |  | 
model.audio_tower.layers.{0...11}.self_attn.k_proj.input_max                 | UNEXPECTED |  | 
model.vision_tower.encoder.layers.{0...15}.self_attn.q_proj.input_max        | UNEXPECTED |  | 
model.vision_tower.encoder.layers.{0...15}.mlp.gate_proj.output_max          | UNEXPECTED |  | 
model.vision_tower.encoder.layers.{0...15}.self_attn.q_proj.output_min       | UNEXPECTED |  | 
model.vision_tower.encoder.layers.{0...15}.mlp.gate_proj.linear.weight       | UNEXPECTED |  | 
model.vision_tower.encoder.layers.{0...15}.post_attention_layernorm.weight   | UNEXPECTE

Unsloth: Not offloading embeddings; offloading is unsupported on WSL.


In [3]:
if not getattr(model.config, "architectures", None):
    model.config.architectures = [type(model).__name__]
print(model.config.architectures)   # ['Gemma4ForCausalLM'] 처럼 나오면 됨

['Gemma4ForCausalLM']


In [4]:
# 층별 임베딩(PLE)을 CPU로 옮기기: 학습하지 않는 조회용 표라 CPU에 둬도 됨
import types, torch.nn.functional as F
PLE_SAVED = []   # GGUF 내보내기 전에 원래대로 되돌릴 때 씀
for n, emb in [(n, m) for n, m in model.named_modules() if n.endswith("embed_tokens_per_layer")]:
    w_cpu = emb.weight.detach().to("cpu")                       # CPU 복사본
    scale = getattr(emb, "embed_scale", None)
    scale = scale.detach().to("cpu") if torch.is_tensor(scale) else scale
    def fwd(self, input_ids, _w = w_cpu, _s = scale):
        out = F.embedding(input_ids.to("cpu"), _w, padding_idx = self.padding_idx)
        if _s is not None: out = out * (_s.to(out.dtype) if torch.is_tensor(_s) else _s)
        return out.to("cuda", non_blocking = True)
    emb.forward = types.MethodType(fwd, emb)                     # 조회를 CPU에서
    PLE_SAVED.append((emb, w_cpu))
    emb.weight = torch.nn.Parameter(torch.empty(0, dtype = w_cpu.dtype, device = "cuda"), requires_grad = False)  # GPU 원본 비우기
    print("옮김:", n, tuple(w_cpu.shape))
torch.cuda.empty_cache()
print("GPU 사용 중:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

옮김: model.embed_tokens_per_layer (262144, 10752)
GPU 사용 중: 4.08 GB


In [5]:
import gc
print("실제 사용(allocated):", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("잡아 둔 양(reserved):", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")
big = [(tuple(o.shape), str(o.dtype), round(o.numel() * o.element_size() / 1024**3, 2))
       for o in gc.get_objects() if torch.is_tensor(o) and o.is_cuda and o.numel() * o.element_size() > 5e8]
print("GPU에 있는 0.5GB 넘는 텐서:", big)

실제 사용(allocated): 4.08 GB
잡아 둔 양(reserved): 9.61 GB
GPU에 있는 0.5GB 넘는 텐서: [((262144, 2560), 'torch.bfloat16', 1.25)]


## 3. 평가 함수
- 정답 판정(기획안 8-4 ①): 같은 메모 안에서 유형(type)과 있었음/없었음(status)이 같으면 맞은 사건
- 함께 보는 것: JSON 형식 통과율, 잡담 메모(정답 빈 결과)에서 빈 결과를 낸 비율, time_expr 일치율
- 결과는 `ml/train/out/eval_<이름>.json`에 저장 (발표 숫자로 씀)

In [6]:
import re
from collections import Counter

def parse_events(text):
    """모델 답에서 JSON을 찾아 events 목록을 돌려줌. 못 읽으면 None"""
    t = text.strip().replace("```json", "").replace("```", "")
    a, b = t.find("{"), t.rfind("}")
    if a < 0 or b < 0: return None
    try:
        obj = json.loads(t[a:b + 1])
        ev = obj.get("events") if isinstance(obj, dict) else None
        return ev if isinstance(ev, list) else None
    except Exception:
        return None

def generate(msgs, max_new_tokens = 384):
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt = True, tokenize = True,
                                           return_dict = True, return_tensors = "pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens = max_new_tokens, do_sample = False)
    return tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens = True)[0]

def evaluate(file, tag, n = None, show = 3):
    rows = [json.loads(l) for l in open(DATA / file, encoding = "utf-8") if l.strip()][:n]
    FastModel.for_inference(model)
    tp = fp = fn = 0; json_ok = 0; empty_total = empty_ok = 0; te_match = te_total = 0
    per_type = {}
    records = []
    t0 = time.time()
    for i, r in enumerate(rows):
        if "messages" in r:   # train/val 형식
            msgs = r["messages"][:2]; gold = json.loads(r["messages"][2]["content"])["events"]
        else:                 # synth_eval 형식
            system = open(ROOT / "config" / "system_prompt.txt", encoding = "utf-8").read().strip()
            msgs = [{"role": "system", "content": system}, {"role": "user", "content": r["memo"]}]
            gold = r["gold"]["events"]
        raw = generate(msgs)
        pred = parse_events(raw)
        if pred is not None: json_ok += 1
        pred = pred or []
        g = Counter((e["type"], e["status"]) for e in gold)
        p = Counter((e.get("type"), e.get("status")) for e in pred if isinstance(e, dict))
        hit = g & p
        tp += sum(hit.values()); fp += sum((p - g).values()); fn += sum((g - p).values())
        for (t, _), c in g.items(): per_type.setdefault(t, [0, 0, 0]); per_type[t][2] += c
        for (t, _), c in p.items(): per_type.setdefault(t, [0, 0, 0]); per_type[t][1] += c
        for (t, _), c in hit.items(): per_type[t][0] += c
        if not gold:
            empty_total += 1; empty_ok += int(len(pred) == 0)
        pd = {(e.get("type"), e.get("status")): e.get("time_expr") for e in pred if isinstance(e, dict)}
        for e in gold:
            k = (e["type"], e["status"])
            if k in pd: te_total += 1; te_match += int(pd[k] == e["time_expr"])
        records.append({"memo": msgs[1]["content"], "gold": gold, "pred_raw": raw})
        if i < show: print(f"[{i}] 메모: {msgs[1]['content']}\n    예측: {raw.strip()[:300]}\n    정답: {json.dumps(gold, ensure_ascii = False)[:300]}\n")
        if (i + 1) % 25 == 0: print(f"  {i + 1}/{len(rows)}건, {round(time.time() - t0)}초")
    prec = tp / (tp + fp) if tp + fp else 0; rec = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0
    res = {"tag": tag, "file": file, "n": len(rows), "precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4),
           "json_ok_rate": round(json_ok / len(rows), 4),
           "empty_correct_rate": round(empty_ok / empty_total, 4) if empty_total else None,
           "time_expr_match_rate": round(te_match / te_total, 4) if te_total else None,
           "per_type_f1": {t: round(2 * v[0] / (v[1] + v[2]), 3) if v[1] + v[2] else None for t, v in sorted(per_type.items())},
           "sec_per_memo": round((time.time() - t0) / len(rows), 2)}
    json.dump({"summary": res, "records": records}, open(OUT / f"eval_{tag}.json", "w", encoding = "utf-8"), ensure_ascii = False, indent = 1)
    print(json.dumps(res, ensure_ascii = False, indent = 1))
    return res

In [7]:
print(generate([{"role": "user", "content": "안녕하세요. 한 문장으로 자기소개 해 주세요."}], max_new_tokens = 40))

Both `max_new_tokens` (=40) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/aiuser/projects/itda-ai/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/home/aiuser/projects/itda-ai/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/home/aiuser/projects/itda-ai/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/home/aiuser/projects/itda-ai/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mkldnn' is deprecated, pl

저는 Google DeepMind에서 개발한 오픈 가중치 대규모 언어 모델인 Gemma 4입니다.


## 4. 파인튜닝 전 평가 (synth_eval 300)
시스템 프롬프트만 주고 베이스 모델이 얼마나 하는지 봄.

In [8]:
# before_test = evaluate("synth_eval.jsonl", tag = "before_test")

In [9]:
before_test = json.load(open(OUT / "eval_before_test.json", encoding = "utf-8"))["summary"]
print(before_test["f1"], before_test["json_ok_rate"])

0.8622 1.0


## 5. LoRA 어댑터 (r, alpha 16)
메모리 오류가 나면 8, 8.

In [10]:
try: FastModel.for_training(model)
except Exception: pass
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none", random_state = 3407,
)

## 6. 데이터

In [11]:
from datasets import load_dataset
train_ds = load_dataset("json", data_files = str(DATA / "train.jsonl"), split = "train")
val_ds   = load_dataset("json", data_files = str(DATA / "val.jsonl"),   split = "train")

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize = False, add_generation_prompt = False).removeprefix("<bos>")
                     for m in batch["messages"]]}

train_ds = train_ds.map(to_text, batched = True, remove_columns = train_ds.column_names)
val_ds   = val_ds.map(to_text,   batched = True, remove_columns = val_ds.column_names)
print(len(train_ds), len(val_ds))   # 1977 200
print(train_ds[0]["text"])

tok = getattr(tokenizer, "tokenizer", tokenizer)
lens = [len(tok(t)["input_ids"]) for t in train_ds["text"]]
print("토큰 최대", max(lens), "평균", round(sum(lens) / len(lens)), "1024 초과", sum(l > 1024 for l in lens))

1977 200
<|turn>system
너는 치매 환자 보호자가 쓴 관찰 메모에서 사건을 뽑아 {"events": [...]} 형태의 JSON으로만 답한다.

[유형] 아래 12개 코드 중 하나만 쓴다.
night_waking: 야간 각성. 밤에 깨서 다시 잠들지 못하거나 밤에 활동함
wandering_exit: 배회·출입문 시도. 목적 없이 돌아다니거나 서성이거나 밖으로 나가려 함
agitation: 초조·공격. 제자리에서 소리 지르기, 밀치기, 때리기 같은 행동적 흥분
irritability: 과민·짜증. 사소한 일에 쉽게 화내거나 짜증을 냄
anxiety: 불안. 걱정하거나 두려워하고, 혼자 있기를 싫어함
low_mood_apathy: 우울·무기력. 슬퍼하거나 처져 있고 의욕·관심이 줄어듦
delusion: 망상. 사실이 아닌 것을 사실로 믿음
hallucination: 환각. 없는 것을 보거나 들음
reduced_intake: 식사량 감소. "평소보다", "~밖에", "몇 숟갈만"처럼 줄었다는 표현이 있거나 식사를 거부함
medication_refusal: 복약 거부. 환자가 약 먹기를 거부하거나 미룸
confusion: 사람·장소 혼동. 사람을 잘못 알아보거나 시간·장소를 헷갈림
fall: 낙상. 넘어지거나 주저앉음

[필드]
- type: 위 코드 중 하나
- status: "present"(있었음) 또는 "absent"(없었음). absent는 메모가 명시적으로 부정할 때만 쓴다
- time_expr: 메모에 쓰인 시간 표현을 그대로(예: "어젯밤"). 없으면 null
- count: 횟수. 언급이 없거나 불분명하면 1
- evidence: 근거가 된 메모 구절을 고치지 않고 그대로 옮긴다

[규칙]
- 메모에 언급된 유형만 출력한다. 언급되지 않은 유형은 출력하지 않는다
- 사건이 없으면 {"events": []}를 출력한다
- 사건은 evidence가 메모에 나오는 순서대로 출력한다
- 밤에 깨서 돌아다니면 night_waking과 wande

## 7. 학습 설정
1977건 × 2에폭 ÷ 8 = 약 494스텝. 100스텝마다 val 손실 확인, 체크포인트 저장.

In [12]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    train_dataset = train_ds, eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10, num_train_epochs = 2, learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "steps", eval_steps = 100,
        save_strategy = "steps", save_steps = 100, save_total_limit = 2,
        output_dir = str(OUT / "checkpoints"),
        optim = "adamw_8bit", weight_decay = 0.001, lr_scheduler_type = "linear",
        seed = 3407, report_to = "none",
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
    ),
)
trainer = train_on_responses_only(trainer)
# 확인: 학습 대상은 JSON 답만이어야 함
print(tok.decode([x for x in trainer.train_dataset[0]["labels"] if x != -100]))

Unsloth: reducing dataset_num_proc 8 -> 2 to fit free memory (~1GB per worker). Set UNSLOTH_DATASET_NUM_PROC to override.
🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Unsloth: Auto-detected instruction_part = '<|turn>user\n' and response_part = '<|turn>model\n'
{"events": [{"type": "irritability", "status": "present", "time_expr": null, "count": 1, "evidence": "괜히 짜증 내심"}, {"type": "wandering_exit", "status": "present", "time_expr": "2일 전", "count": 2, "evidence": "2일 전에는 현관문 열고 나가시려는 걸 두 번 말림"}, {"type": "night_waking", "status": "present", "time_expr": "아까", "count": 3, "evidence": "아까 주무시다가 세 번이나 깨심"}]}<turn|>



## 8. 학습
끊겼다면 1, 2, 5, 6, 7을 다시 실행하고 `trainer.train(resume_from_checkpoint = True)`.

In [13]:
t0 = time.time()
trainer_stats = trainer.train()
print(f"학습 {round((time.time() - t0) / 60, 1)}분, 최대 VRAM {round(torch.cuda.max_memory_reserved() / 1024**3, 2)}GB")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,977 | Num Epochs = 2 | Total steps = 496
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 36,700,160 of 4,736,196,864 (0.77% trained)


Step,Training Loss,Validation Loss
100,0.016072,0.016290
200,0.016759,0.013553
300,0.007375,0.013536
400,0.008207,0.011994
496,0.004711,0.011909


Filter: 100%|██████████| 200/200 [00:00<00:00, 3481.37 examples/s]
Unsloth: Restored added_tokens_decoder metadata in /home/aiuser/projects/itda-ai/ml/train/out_gemma_e4B/checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/aiuser/projects/itda-ai/ml/train/out_gemma_e4B/checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/aiuser/projects/itda-ai/ml/train/out_gemma_e4B/checkpoints/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/aiuser/projects/itda-ai/ml/train/out_gemma_e4B/checkpoints/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/aiuser/projects/itda-ai/ml/train/out_gemma_e4B/checkpoints/checkpoint-496/tokenizer_config.json.


학습 52.9분, 최대 VRAM 9.98GB


## 9. LoRA 저장 (먼저)

In [14]:
model.save_pretrained(str(OUT / "lora_itda"))
tokenizer.save_pretrained(str(OUT / "lora_itda"))
print(os.listdir(OUT / "lora_itda"))

Unsloth: Restored added_tokens_decoder metadata in /home/aiuser/projects/itda-ai/ml/train/out_gemma_e4B/lora_itda/tokenizer_config.json.


['README.md', 'chat_template.jinja', 'adapter_model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'adapter_config.json']


## 10. 파인튜닝 후 평가

In [15]:
after_test = evaluate("synth_eval.jsonl", tag = "after_test")

Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[0] 메모: 우리 양반이 며칠 전에는 문 열고 나갈라 카는 거 없었다. 근데 별것도 아인 걸로 성을 세 번이나 내데. 약 묵으라 카니 안 묵는다꼬 손으로 밀어내삤다 아이가.
    예측: {"events": [{"type": "wandering_exit", "status": "absent", "time_expr": "며칠 전", "count": 1, "evidence": "며칠 전에는 문 열고 나갈라 카는 거 없었다"}, {"type": "irritability", "status": "present", "time_expr": null, "count": 3, "evidence": "별것도 아인 걸로 성을 세 번이나 내데"}, {"type": "medication_refusal", "status": "present", 
    정답: [{"type": "wandering_exit", "status": "absent", "time_expr": "며칠 전", "count": 1, "evidence": "며칠 전에는 문 열고 나갈라 카는 거 없었다"}, {"type": "irritability", "status": "present", "time_expr": null, "count": 3, "evidence": "성을 세 번이나 내데"}, {"type": "medication_refusal", "status": "present", "time_expr": null, "c



Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1] 메모: 할머니 나를 엄마로 두 번 부르심. 약 드시라니까 싫다고 안 드심. 어제 저녁엔 주무시다가 두 번 깨심
    예측: {"events": [{"type": "confusion", "status": "present", "time_expr": null, "count": 2, "evidence": "나를 엄마로 두 번 부르심"}, {"type": "medication_refusal", "status": "present", "time_expr": null, "count": 1, "evidence": "약 드시라니까 싫다고 안 드심"}, {"type": "night_waking", "status": "present", "time_expr": "어제 저녁",
    정답: [{"type": "confusion", "status": "present", "time_expr": null, "count": 2, "evidence": "나를 엄마로 두 번 부르심"}, {"type": "medication_refusal", "status": "present", "time_expr": null, "count": 1, "evidence": "약 드시라니까 싫다고 안 드심"}, {"type": "night_waking", "status": "present", "time_expr": "어제 저녁", "count": 2



Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  250/300건, 2981초


Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  275/300건, 3306초


Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  300/300건, 3608초
{
 "tag": "after_test",
 "file": "synth_eval.jsonl",
 "n": 300,
 "precision": 0.9893,
 "recall": 0.9946,
 "f1": 0.992,
 "json_ok_rate": 1.0,
 "empty_correct_rate": 0.9429,
 "time_expr_match_rate": 0.9856,
 "per_type_f1": {
  "agitation": 0.979,
  "anxiety": 1.0,
  "confusion": 0.989,
  "delusion": 1.0,
  "fall": 1.0,
  "hallucination": 0.989,
  "irritability": 0.989,
  "low_mood_apathy": 1.0,
  "medication_refusal": 1.0,
  "night_waking": 1.0,
  "reduced_intake": 0.979,
  "wandering_exit": 0.978
 },
 "sec_per_memo": 12.03
}


## 11. 전후 비교

In [16]:
keys = ["precision", "recall", "f1", "json_ok_rate", "empty_correct_rate", "time_expr_match_rate", "sec_per_memo"]
print(f"{'지표':22}{'파인튜닝 전':>12}{'파인튜닝 후':>12}")
for k in keys:
    print(f"{k:22}{str(before_test[k]):>12}{str(after_test[k]):>12}")
print("\n유형별 F1 (전 → 후)")
for t in sorted(set(before_test["per_type_f1"]) | set(after_test["per_type_f1"])):
    print(f"  {t:20} {before_test['per_type_f1'].get(t)} → {after_test['per_type_f1'].get(t)}")

지표                          파인튜닝 전      파인튜닝 후
precision                   0.8944      0.9893
recall                      0.8321      0.9946
f1                          0.8622       0.992
json_ok_rate                   1.0         1.0
empty_correct_rate          0.9429      0.9429
time_expr_match_rate        0.8197      0.9856
sec_per_memo                 10.01       12.03

유형별 F1 (전 → 후)
  agitation            0.722 → 0.979
  anxiety              0.911 → 1.0
  confusion            0.805 → 0.989
  delusion             0.863 → 1.0
  fall                 0.909 → 1.0
  hallucination        0.867 → 0.989
  irritability         0.672 → 0.989
  low_mood_apathy      0.907 → 1.0
  medication_refusal   0.957 → 1.0
  night_waking         0.911 → 1.0
  reduced_intake       0.946 → 0.979
  wandering_exit       0.891 → 0.978


## 12. GGUF 내보내기 (Q8_0)
이후 `llama-quantize --allow-requantize ... Q4_K_M`으로 줄이고 Ollama에 등록 (가이드 2-7).

In [17]:
for emb, w in PLE_SAVED:
    del emb.forward                                             # 원래 동작으로
    emb.weight = torch.nn.Parameter(w, requires_grad = False)   # 표를 되돌림 (CPU에 둔 채로 저장됨)
print("PLE 복원 완료")

PLE 복원 완료


In [1]:
import os, json, torch
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "ml" / "data" / "train.jsonl").exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
OUT = ROOT / "ml" / "train" / "out_gemma_e4B"

from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name = str(OUT / "lora_itda"),   # 학습한 LoRA (베이스 모델은 자동으로 함께 불러옴)
    max_seq_length = 1024, load_in_4bit = True, text_only = True,
)
if not getattr(model.config, "architectures", None):
    model.config.architectures = [type(model).__name__]
print("불러오기 완료")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/aiuser/projects/itda-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0929 09:31:49.510000 17899 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0929 09:31:49.549000 17899 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/home/aiuser/projects/itda-ai/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:4896: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)
[unsloth_zoo.log|WARNING]Loading /home/aiuser/projects/itda-ai/ml/train/out_gemma_e4B/lora_itda as text-only; vision/audio towers skipped. Use FastVisionModel for multimodal inputs.


==((====))==  Unsloth 2026.9.11: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4070 Ti. Num GPUs = 1. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not planning a device map; text_only loads a decoder the repo config does not describe. Using `sequential`.


Loading weights: 100%|██████████| 719/719 [00:05<00:00, 139.17it/s]
Gemma4ForCausalLM LOAD REPORT from: unsloth/gemma-4-E4B-it-unsloth-bnb-4bit
Key                                                                          | Status     |  | 
-----------------------------------------------------------------------------+------------+--+-
model.vision_tower.encoder.layers.{0...15}.self_attn.o_proj.input_min        | UNEXPECTED |  | 
model.vision_tower.encoder.layers.{0...15}.mlp.up_proj.linear.weight         | UNEXPECTED |  | 
model.vision_tower.encoder.layers.{0...15}.mlp.gate_proj.linear.weight       | UNEXPECTED |  | 
model.audio_tower.layers.{0...11}.feed_forward1.ffw_layer_1.input_min        | UNEXPECTED |  | 
model.vision_tower.encoder.layers.{0...15}.self_attn.q_proj.input_max        | UNEXPECTED |  | 
model.vision_tower.encoder.layers.{0...15}.self_attn.v_proj.input_max        | UNEXPECTED |  | 
model.audio_tower.layers.{0...11}.lconv1d.depthwise_conv1d.weight            | UNEXPECTE

불러오기 완료


In [3]:
model.save_pretrained_gguf(str(OUT / "gguf"), tokenizer, quantization_method = "Q8_0")
print([str(p) for p in OUT.rglob("*.gguf")])

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /home/aiuser/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/home/aiuser/projects/itda-ai/ml/train/out_gemma_e4B/gguf`: 100%|██████████| 1/1 [00:21<00:00, 21.47s/it]


Successfully copied all 1 files from cache to `/home/aiuser/projects/itda-ai/ml/train/out_gemma_e4B/gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]


RuntimeError: Failed to save/merge model: RuntimeError: Model merge failed with error: [Errno 13] Permission denied: '/home/aiuser/projects/itda-ai/ml/train/out_gemma_e4B/gguf/model.safetensors'

## 13. (선택) 합성 평가 세트 300건 - 발표용 숫자
학습에 쓰지 않은 세트. 파인튜닝 후 모델로 잼. 파인튜닝 전 숫자는 Ollama 베이스라인(가이드 2-4)이나, 학습 전에 4번 셀 옆에서 `evaluate("synth_eval.jsonl", "before_test")`를 먼저 돌려 두면 같은 조건으로 비교할 수 있음.

In [ ]:
# after_test = evaluate("synth_eval.jsonl", tag = "after_test")